# SmartKart — AI Customer Support & Order Resolution Assistant

LangChain + Google Gemini. Each section below is one module of the problem statement.

**Setup, once:**
1. `pip install -r requirements.txt`
2. Get a free key at [aistudio.google.com/apikey](https://aistudio.google.com/apikey)
3. Paste it into the `.env` file next to this notebook

---
## Module 1 — Connect LangChain with Google Gemini

Goal: prove the pipe works. One question in, one natural-language answer out.

The API key is **never hard-coded**. It lives in `.env`, gets loaded into the environment by `python-dotenv`, and is read back with `os.getenv`.

In [1]:
import os

from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

# Reads .env into the process environment. The key itself never appears in code.
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
assert GOOGLE_API_KEY, "GOOGLE_API_KEY is not set — paste your key into the .env file."

print("API key loaded:", GOOGLE_API_KEY[:6] + "..." + GOOGLE_API_KEY[-4:])

API key loaded: AIzaSy...ygRk


In [3]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    # Low temperature: a support bot should be factual and consistent,
    # not creative. 0.0-0.3 is the right band here.
    temperature=0.2,
    google_api_key=GOOGLE_API_KEY,
)

llm

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13', 'langchain-google-genai': '4.2.7'}}, output_version=None, profile={'name': 'Gemini 3.1 Flash Lite', 'release_date': '2026-05-07', 'last_updated': '2026-05-07', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-3.1-flash-lite', temperature=0.2, client=<google.genai.client.Client object at 0x00000135B1E39550>, default_metadata=(), model_kwargs={})

In [4]:
response = llm.invoke("Explain customer-support automation in three sentences.")

# Use .text, not .content — Gemini 3.x returns .content as a list of content
# blocks, while .text gives the plain string.
print(response.text)

Customer-support automation uses software, such as chatbots and AI-driven ticketing systems, to handle routine inquiries and tasks without human intervention. By instantly resolving common questions and routing complex issues to the appropriate agents, it significantly reduces response times and operational costs. This technology allows support teams to focus on high-value interactions while ensuring customers receive 24/7 assistance.


`.invoke()` returns an `AIMessage`, not a plain string.

Two things worth noticing on it:
- **`.text`** is the plain answer. (`.content` is the raw form — on Gemini 3.x it's a *list* of content blocks, so printing it looks messy.)
- **`.tool_calls`** is empty, because no tools are bound yet. From Module 5 onward, that exact field is what drives the whole assistant.

In [ ]:
print("type:      ", type(response).__name__)
print("text:      ", response.text[:60], "...")
print("tool_calls:", response.tool_calls)  # empty — no tools bound yet
print("tokens:    ", response.usage_metadata)